# Lab record

The lab's index: every hypothesis with its status and decision, every run with its config, data, metrics and card,
and the standard report for any of them.

- The registry and runs live in `quantlab_data/research/lab.db` (SQLite; separate from the trading database).
- Seed the registry from this project's pre-registrations once: `python scripts/lab/seed_registry.py`.
- Reports are self-contained HTML in `quantlab_data/reports/` (open in a browser, print to PDF if needed).

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from quantlab.data import align
from quantlab.data.quality import data_card
from quantlab.signals import library as lib
from quantlab.engine.components import Targets, apply_signal
from quantlab.engine.backtest import run_backtest, Sizer, EngineConfig
from quantlab.regimes.models import PercentileRegime, labels
from quantlab.reporting import charts
from quantlab.reporting.report import build_report
from quantlab.tracking.store import ExperimentStore
from quantlab.validation.strategy_card import card_backtest

charts.apply_style()
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
store = ExperimentStore()

## 1. The lab's state

Active work first. `store.history("H13")` shows how a hypothesis got to its status.

In [ ]:
store.lab_state()[["status", "title", "runs", "latest verdict", "decision"]]

## 2. Recording runs

Wrap a study in `store.run(...)`: its config, data snapshots and commit are recorded, and whatever it logs (metrics,
cards, series, figures, reports) is indexed. A run that raises is recorded as failed, with the error.

In [ ]:
store.register("DEMO", "Lab record walkthrough: H14's trend model on NDX at three speeds", status="idea")

bars = pd.read_csv("../data/research_cache/h14/NDX.csv", index_col=0, parse_dates=True)
bars.columns = [c.lower() for c in bars.columns]
panel = align({"NDX": bars})
returns = bars["close"].pct_change().to_frame("NDX")

def run(fast=20, slow=100, cost_multiplier=1.0):
    return run_backtest(Targets(apply_signal(lib.h14_trend(fast=fast, slow=slow), panel)), returns, costs_bps=1.5,
                        sizer=Sizer(0.10, span=20, cap=2.0), config=EngineConfig(band=0.02, max_position=2.0, cost_multiplier=cost_multiplier))

results = {}
for fast, slow in [(10, 50), (20, 100), (40, 200)]:
    config = {"signal": {"name": "h14_trend", "fast": fast, "slow": slow}, "sizer": {"target_vol": 0.10, "span": 20, "cap": 2.0}, "cost_bps": 1.5}
    with store.run("DEMO", f"trend {fast}/{slow}", config=config, snapshots=["research_cache/h14/NDX.csv"]) as record:
        results[(fast, slow)] = result = run(fast, slow)
        record.log_result(result)
        record.log_card(card_backtest(result, name=f"DEMO trend {fast}/{slow}", family="DEMO"))
store.runs("DEMO", metrics=("Sharpe", "annual return", "max drawdown", "turnover"))

In [ ]:
ids = list(store.runs("DEMO").index[-3:])
store.compare(ids, keys=["Sharpe", "annual return", "max drawdown", "turnover", "card shrunk_sharpe", "card placebo_p"])

## 3. Charts

The same charts the reports use, inline. Regime shading takes any labels.

In [ ]:
primary = results[(20, 100)]
charts.tearsheet(primary.pnl, benchmark=returns["NDX"], labels=labels(PercentileRegime("er")(bars)).loc["2015":], title="H14 trend on NDX")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
charts.equity({f"{f}/{s}": r for (f, s), r in results.items()}, ax=axes[0])
charts.attribution(primary, ax=axes[1])
plt.tight_layout(); plt.show()
charts.monthly_returns(primary.pnl.loc["2010":]); plt.show()

## 4. The standard report

One call. The first section is written for a portfolio manager: verdict, what it is, the numbers, what can go wrong,
and the recommendation that follows. The report is logged as an artefact of a run.

In [ ]:
hold = run_backtest(Targets(pd.DataFrame(1.0, index=bars.index, columns=["NDX"])), returns, costs_bps=1.5,
                    sizer=Sizer(0.10, span=20, cap=2.0), config=EngineConfig(band=0.02, max_position=2.0))
with store.run("DEMO", "report: trend 20/100", config={"fast": 20, "slow": 100}) as record:
    path = build_report(primary, name="H14 trend on NDX", hypothesis="DEMO", baseline=hold, benchmark=returns["NDX"],
                        definition="H14's trend model alone: 0.5 tanh of the 20/100-day moving-average gap over ATR plus 0.5 x a "
                                   "60/20-day breakout state, sized to 10% volatility, trading at the next close.",
                        config={"fast": 20, "slow": 100, "entry": 60, "exit": 20, "target_vol": 0.10, "cost_bps": 1.5},
                        data_cards=[data_card(bars, name="NDX")], labels=labels(PercentileRegime("er")(bars)),
                        cost_run=lambda m: run(cost_multiplier=m), snapshots=["research_cache/h14/NDX.csv"])
    record.log_artefact("report", path)
display(HTML(f'<a href="{path.resolve().as_uri()}" target="_blank">open the report: {path}</a>'))

## 5. Updating a hypothesis

A status changes with a reason; the decision text is what the lab state shows.

```python
store.set_status("DEMO", "failed", reason="card: weak (timing placebo 0.58, loses to buy-and-hold)",
                 decision="Trend alone on NDX earns the index drift; not a timing edge.")
```